# Parte 3 — Nova visão: agregações e features

**Objetivo desta parte:** esta é a parte de "manipulação de dados" propriamente dita.
Partindo do dado tratado em granularidade horária (Parte 2), vamos construir uma
**visão diária** e uma série de features derivadas, praticando `groupby`,
`resample`, `rolling`, `pivot_table`, `merge`, `apply` e funções `lambda`.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

PROCESSED_DIR = Path("../data/processed")

df = pd.read_csv(PROCESSED_DIR / "clima_tratado.csv", parse_dates=["datetime"])
df.head()


,cidade,datetime,temp_c,umidade_pct,precipitacao_mm,vento_kmh
0,manaus,2025-01-01 00:00:00,26.7,90.0,0.0,6.4
1,manaus,2025-01-01 01:00:00,26.6,91.0,0.0,5.6
2,manaus,2025-01-01 02:00:00,26.6,92.0,0.0,6.5
3,manaus,2025-01-01 03:00:00,26.5,93.0,0.0,6.1
4,manaus,2025-01-01 04:00:00,25.7,94.0,0.0,2.5


## 1. Visão diária com `groupby` + `resample`

Duas formas de chegar no mesmo resultado — média/mín/máx de temperatura por
cidade/dia — que vale comparar:

- `groupby(["cidade", pd.Grouper(freq="D", key="datetime")])`: agrupa por cidade e
  por dia de calendário em uma única chamada.
- `resample("D")` **por grupo**: mais natural quando você já está trabalhando
  dentro de uma série temporal de uma cidade só.

Usamos a primeira abordagem como caminho principal, por deixar explícito que
`cidade` é uma dimensão de agrupamento igual a `datetime`.


In [2]:
diario = (
    df.groupby(["cidade", pd.Grouper(key="datetime", freq="D")])
    .agg(
        temp_media=("temp_c", "mean"),
        temp_min=("temp_c", "min"),
        temp_max=("temp_c", "max"),
        umidade_media=("umidade_pct", "mean"),
        precipitacao_total=("precipitacao_mm", "sum"),
        vento_medio=("vento_kmh", "mean"),
    )
    .reset_index()
    .rename(columns={"datetime": "data"})
)
diario["data"] = diario["data"].dt.date
diario.head()


,cidade,data,temp_media,temp_min,temp_max,umidade_media,precipitacao_total,vento_medio
0,manaus,2025-01-01,27.391667,25.5,29.8,84.125000,11.0,5.654167
1,manaus,2025-01-02,25.908333,24.7,27.8,86.104167,7.3,6.083333
2,manaus,2025-01-03,27.587500,24.9,30.2,80.916667,1.0,4.937500
3,manaus,2025-01-04,26.233333,24.5,28.9,86.166667,11.8,4.608333
4,manaus,2025-01-05,25.795833,24.3,28.0,86.916667,8.8,4.100000


In [3]:
# Mesma agregação via resample, para uma única cidade — útil para conferir
# que os dois caminhos batem.
uma_cidade = df[df["cidade"] == "sao_paulo"].set_index("datetime")
conferencia_resample = uma_cidade["temp_c"].resample("D").agg(["mean", "min", "max"])
conferencia_resample.head()


,mean,min,max
datetime,,,
2025-01-01,22.329167,18.1,28.5
2025-01-02,23.362500,19.0,28.6
2025-01-03,22.550000,19.4,27.6
2025-01-04,22.350000,18.6,27.1
2025-01-05,22.500000,17.3,29.6


## 2. Classificação categórica com `apply`/`lambda`

Duas classificações simples baseadas em limiares — um bom exemplo de quando vale a
pena escrever uma função nomeada (`classificar_temperatura`, reutilizável e
testável) versus uma `lambda` de uma linha (`classificar_chuva`, curta o
suficiente para não precisar de nome).


In [4]:
def classificar_temperatura(temp_media: float) -> str:
    if temp_media < 18:
        return "frio"
    elif temp_media < 26:
        return "ameno"
    else:
        return "quente"


diario["categoria_temp"] = diario["temp_media"].apply(classificar_temperatura)

diario["categoria_chuva"] = diario["precipitacao_total"].apply(
    lambda mm: "chuvoso" if mm > 1.0 else "seco"
)

diario[["cidade", "data", "temp_media", "categoria_temp", "precipitacao_total", "categoria_chuva"]].head()


,cidade,data,temp_media,categoria_temp,precipitacao_total,categoria_chuva
0,manaus,2025-01-01,27.391667,quente,11.0,chuvoso
1,manaus,2025-01-02,25.908333,ameno,7.3,chuvoso
2,manaus,2025-01-03,27.587500,quente,1.0,seco
3,manaus,2025-01-04,26.233333,quente,11.8,chuvoso
4,manaus,2025-01-05,25.795833,ameno,8.8,chuvoso


In [5]:
diario["categoria_temp"].value_counts()


categoria_temp
quente    89
ameno     66
Name: count, dtype: int64

### Versão vetorizada com `np.where` (quando a lógica é simples)

`apply`/`lambda` chama uma função Python **uma vez por linha** — ótimo para lógica
arbitrária, mas mais lento que uma operação vetorizada quando a regra é só uma
comparação booleana. Para condições binárias como `categoria_chuva`,
`np.where(condicao, valor_se_true, valor_se_false)` calcula o array inteiro de uma
vez, sem o overhead do loop Python.

Repetimos a mesma classificação, agora na granularidade horária (`df`, muito mais
linhas que `diario`) para deixar a diferença de desempenho visível.

In [6]:
categoria_chuva_apply = df["precipitacao_mm"].apply(lambda mm: "chuvoso" if mm > 0 else "seco")
categoria_chuva_np_where = np.where(df["precipitacao_mm"] > 0, "chuvoso", "seco")

# Confirma que os dois caminhos concordam antes de comparar velocidade
(categoria_chuva_apply.values == categoria_chuva_np_where).all()

np.True_

In [7]:
%timeit df["precipitacao_mm"].apply(lambda mm: "chuvoso" if mm > 0 else "seco")
%timeit np.where(df["precipitacao_mm"] > 0, "chuvoso", "seco")

328 μs ± 303 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


55.9 μs ± 237 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


Para mais de duas categorias (como `classificar_temperatura`, com
"frio"/"ameno"/"quente"), o equivalente vetorizado é `np.select`, que recebe uma
lista de condições e uma lista de valores correspondentes, avaliadas na ordem — a
primeira condição verdadeira vence:

In [8]:
condicoes = [
    diario["temp_media"] < 18,
    diario["temp_media"] < 26,
]
valores = ["frio", "ameno"]

diario["categoria_temp_np_select"] = np.select(condicoes, valores, default="quente")

(diario["categoria_temp_np_select"] == diario["categoria_temp"]).all()

np.True_

`np.select` é mais rápido, mas menos legível que a função nomeada
`classificar_temperatura` assim que as regras crescem (mais faixas, condições
compostas) — por isso mantemos a versão com `apply` como a oficial em
`diario["categoria_temp"]` daqui para frente; `categoria_temp_np_select` foi só
para conferência, então descartamos a coluna de comparação.

In [9]:
diario = diario.drop(columns=["categoria_temp_np_select"])

## 3. Médias móveis (`rolling`) de 3 e 7 dias

`rolling` precisa de dados **ordenados no tempo** e, como temos 5 cidades
misturadas na mesma tabela, o cálculo tem que ser feito **por grupo**
(`groupby("cidade")`) — senão a janela móvel "vaza" dados de uma cidade para outra
nos primeiros dias de cada cidade seguinte.


In [10]:
diario = diario.sort_values(["cidade", "data"]).reset_index(drop=True)

diario["media_movel_3d"] = diario.groupby("cidade")["temp_media"].transform(
    lambda s: s.rolling(window=3, min_periods=1).mean()
)
diario["media_movel_7d"] = diario.groupby("cidade")["temp_media"].transform(
    lambda s: s.rolling(window=7, min_periods=1).mean()
)

diario[["cidade", "data", "temp_media", "media_movel_3d", "media_movel_7d"]].head(10)


,cidade,data,temp_media,media_movel_3d,media_movel_7d
0,manaus,2025-01-01,27.391667,27.391667,27.391667
1,manaus,2025-01-02,25.908333,26.650000,26.650000
2,manaus,2025-01-03,27.587500,26.962500,26.962500
3,manaus,2025-01-04,26.233333,26.576389,26.780208
4,manaus,2025-01-05,25.795833,26.538889,26.583333
5,manaus,2025-01-06,26.475000,26.168056,26.565278
6,manaus,2025-01-07,26.575000,26.281944,26.566667
7,manaus,2025-01-08,26.900000,26.650000,26.496429
8,manaus,2025-01-09,27.466667,26.980556,26.719048
9,manaus,2025-01-10,27.860417,27.409028,26.758036


## 4. Ranking de cidades por dia

Para cada data, qual foi a cidade mais quente? Usamos `groupby("data")` +
`rank()` para gerar um ranking completo, e também isolamos só o "primeiro
colocado" de cada dia com `idxmax`.


In [11]:
diario["ranking_temp_dia"] = diario.groupby("data")["temp_media"].rank(
    ascending=False, method="min"
).astype(int)

diario[diario["ranking_temp_dia"] == 1][["data", "cidade", "temp_media"]].head(10)


,data,cidade,temp_media
0,2025-01-01,manaus,27.391667
9,2025-01-10,manaus,27.860417
11,2025-01-12,manaus,27.400000
12,2025-01-13,manaus,28.025000
13,2025-01-14,manaus,27.862500
14,2025-01-15,manaus,27.752083
15,2025-01-16,manaus,28.066667
26,2025-01-27,manaus,26.950000
27,2025-01-28,manaus,27.383333
53,2025-01-23,porto_alegre,28.437500


In [12]:
# Cidade mais quente de cada dia, em uma tabela compacta (um registro por dia)
mais_quente_do_dia = (
    diario.loc[diario.groupby("data")["temp_media"].idxmax()]
    [["data", "cidade", "temp_media"]]
    .rename(columns={"cidade": "cidade_mais_quente"})
    .reset_index(drop=True)
)
mais_quente_do_dia.head(10)


,data,cidade_mais_quente,temp_media
0,2025-01-01,manaus,27.391667
1,2025-01-02,recife,26.954167
2,2025-01-03,rio_de_janeiro,28.950000
3,2025-01-04,rio_de_janeiro,28.491667
4,2025-01-05,recife,27.452083
5,2025-01-06,recife,27.016667
6,2025-01-07,recife,27.250000
7,2025-01-08,recife,27.504167
8,2025-01-09,recife,28.070833
9,2025-01-10,manaus,27.860417


## 5. Tabela pivotada

Linhas = data, colunas = cidade, valores = temperatura média — o formato "largo"
que vamos reaproveitar depois no gráfico comparativo do Streamlit e também salvar
aqui como uma view própria.


In [13]:
pivot_temp = pd.pivot_table(
    diario, index="data", columns="cidade", values="temp_media"
)
pivot_temp.head()


cidade,manaus,porto_alegre,recife,rio_de_janeiro,sao_paulo
data,,,,,
2025-01-01,27.391667,25.083333,26.989583,25.270833,22.329167
2025-01-02,25.908333,26.625000,26.954167,26.491667,23.362500
2025-01-03,27.587500,24.679167,27.191667,28.950000,22.550000
2025-01-04,26.233333,23.158333,27.208333,28.491667,22.350000
2025-01-05,25.795833,22.606250,27.452083,25.475000,22.500000


## 6. `merge` com metadados de cidade

Criamos uma tabela pequena de metadados (região e UF de cada cidade) e fazemos um
`merge` com a visão diária — o cenário clássico de enriquecer uma tabela de fatos
com uma tabela de dimensão.


In [14]:
cidades_meta = pd.DataFrame([
    {"cidade": "sao_paulo", "nome_exibicao": "São Paulo", "uf": "SP", "regiao": "Sudeste"},
    {"cidade": "rio_de_janeiro", "nome_exibicao": "Rio de Janeiro", "uf": "RJ", "regiao": "Sudeste"},
    {"cidade": "manaus", "nome_exibicao": "Manaus", "uf": "AM", "regiao": "Norte"},
    {"cidade": "porto_alegre", "nome_exibicao": "Porto Alegre", "uf": "RS", "regiao": "Sul"},
    {"cidade": "recife", "nome_exibicao": "Recife", "uf": "PE", "regiao": "Nordeste"},
])

diario = diario.merge(cidades_meta, on="cidade", how="left")
diario.head()


,cidade,data,temp_media,temp_min,temp_max,umidade_media,precipitacao_total,vento_medio,categoria_temp,categoria_chuva,media_movel_3d,media_movel_7d,ranking_temp_dia,nome_exibicao,uf,regiao
0,manaus,2025-01-01,27.391667,25.5,29.8,84.125000,11.0,5.654167,quente,chuvoso,27.391667,27.391667,1,Manaus,AM,Norte
1,manaus,2025-01-02,25.908333,24.7,27.8,86.104167,7.3,6.083333,ameno,chuvoso,26.650000,26.650000,4,Manaus,AM,Norte
2,manaus,2025-01-03,27.587500,24.9,30.2,80.916667,1.0,4.937500,quente,seco,26.962500,26.962500,2,Manaus,AM,Norte
3,manaus,2025-01-04,26.233333,24.5,28.9,86.166667,11.8,4.608333,quente,chuvoso,26.576389,26.780208,3,Manaus,AM,Norte
4,manaus,2025-01-05,25.795833,24.3,28.0,86.916667,8.8,4.100000,ameno,chuvoso,26.538889,26.583333,2,Manaus,AM,Norte


In [15]:
# Com a região disponível, dá pra responder perguntas de grupo mais amplas,
# ex.: temperatura média por região (não só por cidade)
diario.groupby("regiao")["temp_media"].mean().sort_values(ascending=False)


regiao
Norte       27.070766
Nordeste    26.910820
Sul         25.181048
Sudeste     24.866196
Name: temp_media, dtype: float64

## 7. Índice de conforto térmico

Uma versão simplificada de "sensação térmica" combinando temperatura e umidade
(inspirada no *heat index*): a partir de ~27°C, umidade alta faz a sensação subir
acima da temperatura real; abaixo disso, a fórmula completa do heat index perde
sentido físico, então mantemos a temperatura como está.

Aplicamos a função **linha a linha** com `apply(axis=1)` — mais lento que uma
versão vetorizada, mas bem mais legível, o que vale a pena para uma fórmula usada
apenas na agregação diária (poucas centenas de linhas).


In [16]:
def calcular_indice_conforto(row) -> float:
    temp = row["temp_media"]
    umidade = row["umidade_media"]

    if temp < 27:
        return round(temp, 1)

    # Versão simplificada do heat index (Rothfusz), em Celsius
    indice = (
        -8.784
        + 1.611 * temp
        + 2.339 * umidade
        - 0.146 * temp * umidade
        + -1.230e-2 * temp ** 2
        + -1.642e-2 * umidade ** 2
        + 2.212e-3 * temp ** 2 * umidade
        + 7.255e-4 * temp * umidade ** 2
        + -3.582e-6 * temp ** 2 * umidade ** 2
    )
    return round(indice, 1)


diario["indice_conforto_c"] = diario.apply(calcular_indice_conforto, axis=1)
diario[["cidade", "data", "temp_media", "umidade_media", "indice_conforto_c"]].sort_values(
    "indice_conforto_c", ascending=False
).head(10)


,cidade,data,temp_media,umidade_media,indice_conforto_c
113,rio_de_janeiro,2025-01-21,31.691667,56.708333,35.7
112,rio_de_janeiro,2025-01-20,30.129167,69.375000,35.5
110,rio_de_janeiro,2025-01-18,30.537500,65.083333,35.4
114,rio_de_janeiro,2025-01-22,30.212500,66.041667,34.9
111,rio_de_janeiro,2025-01-19,29.379167,71.625000,34.2
117,rio_de_janeiro,2025-01-25,30.056250,63.791667,34.0
118,rio_de_janeiro,2025-01-26,29.060417,70.666667,33.3
109,rio_de_janeiro,2025-01-17,28.750000,72.145833,32.9
115,rio_de_janeiro,2025-01-23,28.358333,76.000000,32.7
15,manaus,2025-01-16,28.066667,79.291667,32.5


## 8. Salvando as novas views

Guardamos a visão diária completa (com todas as features desta parte) e a tabela
pivotada — ambas serão a base da Parte de exploração visual e da persistência em
SQLite.


In [17]:
diario.to_csv(PROCESSED_DIR / "clima_diario.csv", index=False)
pivot_temp.to_csv(PROCESSED_DIR / "clima_pivot_temp.csv")

print("Salvos:")
print(" -", PROCESSED_DIR / "clima_diario.csv", diario.shape)
print(" -", PROCESSED_DIR / "clima_pivot_temp.csv", pivot_temp.shape)


Salvos:
 - ../data/processed/clima_diario.csv (155, 17)
 - ../data/processed/clima_pivot_temp.csv (31, 5)
